# Análise do Pipeline KEV

Este notebook é uma camada de análise e validação sobre o pacote local `kev_pipeline`.

O objetivo aqui não é reimplementar o pipeline, mas explorar os artefatos gerados pela execução operacional, validar resultados e facilitar inspeções pontuais.

## O que o projeto faz

- coleta o catálogo KEV da CISA como fonte primária
- normaliza a base canônica de ameaças por CVE
- gera contagens diárias, agregações por vendor e product e um resumo da execução
- mantém histórico local em `artifacts/snapshots/YYYY-MM-DD/`
- gera deltas em `artifacts/deltas/YYYY-MM-DD/` para destacar novos CVEs, itens urgentes e casos com `ransomware_flag=1`
- adiciona NVD e EPSS apenas como enriquecimentos opcionais

## Como usar este notebook

- ajuste a configuração na célula 1
- execute a célula de execução principal para rodar o pacote
- use as seções seguintes para validar métricas, revisar deltas e inspecionar artefatos

## Princípios do projeto

- KEV é a fonte primária
- `dateAdded` é o eixo temporal operacional
- o fluxo principal continua válido em modo `KEV-only`
- enriquecimentos não fazem parte do caminho crítico


In [ ]:
# 1) configuração do notebook e do pipeline
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kev_pipeline import PipelineConfig, run_pipeline

PIPELINE_MODE = "kev"
RUN_NVD = False
RUN_EPSS = False
SNAPSHOT_DATE = None  # ex.: "2026-03-29"
GENERATE_PLOTS = True
NVD_API_KEY = ""

OUT_DIR = ROOT / "artifacts" / "current"
PLOTS_DIR = ROOT / "artifacts" / "plots"
SNAPSHOTS_DIR = ROOT / "artifacts" / "snapshots"
DELTAS_DIR = ROOT / "artifacts" / "deltas"

print(f"Root: {ROOT}")
print(f"Src: {SRC}")
print(f"Pipeline mode: {PIPELINE_MODE}")
print(f"NVD ativo: {RUN_NVD}")
print(f"EPSS ativo: {RUN_EPSS}")


In [ ]:
# 2) helpers de leitura dos artefatos gerados pelo pacote

def load_csv_if_exists(path):
    if not path:
        return pd.DataFrame()

    path = Path(path)
    if not path.exists() or not path.is_file():
        return pd.DataFrame()

    return pd.read_csv(path)


def build_config():
    snapshot_date = None
    if SNAPSHOT_DATE:
        snapshot_date = pd.to_datetime(SNAPSHOT_DATE).date()

    return PipelineConfig(
        pipeline_mode=PIPELINE_MODE,
        run_nvd=RUN_NVD,
        run_epss=RUN_EPSS,
        out_dir=OUT_DIR,
        plots_dir=PLOTS_DIR,
        snapshots_dir=SNAPSHOTS_DIR,
        deltas_dir=DELTAS_DIR,
        nvd_api_key=NVD_API_KEY,
        generate_plots=GENERATE_PLOTS,
        snapshot_date=snapshot_date or PipelineConfig().snapshot_date,
    )


In [ ]:
# 3) execução principal via pacote kev_pipeline
config = build_config()
summary = run_pipeline(config)

summary_path = Path(summary["files"]["summary"])
snapshot_dir = Path(summary["files"]["snapshot_dir"])
delta_dir = Path(summary["files"]["delta_dir"])

threats_daily_events_df = load_csv_if_exists(summary["files"].get("threats_daily_events"))
threats_daily_counts_df = load_csv_if_exists(summary["files"].get("threats_daily_counts"))
threats_by_vendor_df = load_csv_if_exists(summary["files"].get("threats_by_vendor"))
threats_by_product_df = load_csv_if_exists(summary["files"].get("threats_by_product"))
enrich_nvd_df = load_csv_if_exists(summary["files"].get("enrich_nvd"))
enrich_epss_df = load_csv_if_exists(summary["files"].get("enrich_epss"))
threats_daily_enriched_df = load_csv_if_exists(summary["files"].get("threats_daily_enriched"))
new_cves_today_df = load_csv_if_exists(delta_dir / "new_cves_today.csv")
new_urgent_today_df = load_csv_if_exists(delta_dir / "new_urgent_today.csv")
new_ransomware_today_df = load_csv_if_exists(delta_dir / "new_ransomware_today.csv")

print(json.dumps(summary, indent=2, ensure_ascii=False))


## Análise e validação

Esta seção trabalha sobre os artefatos produzidos pelo pacote `kev_pipeline` e serve para verificar a qualidade operacional da execução.

### O que revisar

- volume total de eventos normalizados
- intervalo temporal disponível no histórico
- total de itens urgentes e com `ransomware_flag`
- vendors e products mais frequentes
- delta em relação ao snapshot anterior
- consistência entre `summary.json` e os CSVs gerados


In [ ]:
# 4) análise e validação
if threats_daily_events_df.empty:
    raise RuntimeError("Execute a célula 4 (execução principal) antes desta análise.")

metrics = {
    "rows_threats_daily_events": int(len(threats_daily_events_df)),
    "rows_threats_daily_counts": int(len(threats_daily_counts_df)),
    "unique_days": int(threats_daily_events_df["date"].nunique()) if "date" in threats_daily_events_df.columns else 0,
    "ransomware_flag_sum": int(threats_daily_events_df["ransomware_flag"].sum()) if "ransomware_flag" in threats_daily_events_df.columns else 0,
    "urgent_sum": int(threats_daily_events_df["urgent"].sum()) if "urgent" in threats_daily_events_df.columns else 0,
    "min_date": "" if threats_daily_events_df.empty else str(threats_daily_events_df["date"].min()),
    "max_date": "" if threats_daily_events_df.empty else str(threats_daily_events_df["date"].max()),
    "delta_new_cves_today": int(len(new_cves_today_df)),
    "delta_new_urgent_today": int(len(new_urgent_today_df)),
    "delta_new_ransomware_today": int(len(new_ransomware_today_df)),
}

print("Métricas principais")
print(json.dumps(metrics, indent=2, ensure_ascii=False))

print()
print("Top vendors")
display(threats_by_vendor_df.head(15))

print()
print("Top products")
display(threats_by_product_df.head(15))

print()
print("Novos CVEs do delta")
display(new_cves_today_df.head(10))


In [ ]:
# 5) visualizações rápidas a partir dos CSVs gerados
if threats_daily_counts_df.empty:
    raise RuntimeError("Execute a célula 4 (execução principal) antes dos gráficos.")

import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import HTML, display

fig_daily = px.line(
    threats_daily_counts_df,
    x="date",
    y="threat_count",
    title="Ameaças adicionadas por dia (KEV)",
)
display(HTML(fig_daily.to_html(include_plotlyjs="cdn", full_html=False)))

plt.figure(figsize=(12, 4))
plt.plot(pd.to_datetime(threats_daily_counts_df["date"]), threats_daily_counts_df["threat_count"], linewidth=1.2)
plt.title("Ameaças adicionadas por dia (KEV)")
plt.xlabel("Data")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()


In [ ]:
# 6) resumo final e localização dos artefatos
print("Resumo final")
print(json.dumps(summary, indent=2, ensure_ascii=False))

print()
print("Snapshot atual:")
print(snapshot_dir)

print()
print("Delta atual:")
print(delta_dir)


## Enriquecimento opcional com NVD e EPSS

Esta seção existe para aprofundamento analítico. O pipeline principal não depende desses enriquecimentos para gerar a base canônica e os deltas operacionais.

### Leitura operacional

- se `RUN_NVD=False`, o fluxo principal continua íntegro
- se `RUN_EPSS=False`, os deltas principais continuam íntegros
- se houver falhas de enriquecimento, elas aparecem em `summary["enrichment_failures"]`
- a ausência de enriquecimento não invalida os artefatos de `artifacts/current/`, `artifacts/snapshots/` ou `artifacts/deltas/`


In [ ]:
# 7) validação da seção opcional de enriquecimento
print("Warnings:")
print(json.dumps(summary.get("warnings", []), indent=2, ensure_ascii=False))

print()
print("Falhas de enriquecimento:")
print(json.dumps(summary.get("enrichment_failures", {}), indent=2, ensure_ascii=False))

if not enrich_nvd_df.empty:
    print()
    print(f"NVD enrichment disponível: {len(enrich_nvd_df):,} linhas")
    display(enrich_nvd_df.head(10))
else:
    print()
    print("NVD sem dados nesta execução (ou desativado).")

if not enrich_epss_df.empty:
    print()
    print(f"EPSS enrichment disponível: {len(enrich_epss_df):,} linhas")
    display(enrich_epss_df.head(10))
else:
    print()
    print("EPSS sem dados nesta execução (ou desativado).")

if not threats_daily_enriched_df.empty:
    print()
    print(f"Base enriquecida disponível: {len(threats_daily_enriched_df):,} linhas")
    display(threats_daily_enriched_df.head(10))
else:
    print()
    print("Base enriquecida não foi gerada nesta execução.")
